# Chapter 9b — Parallel Programming in CUDA C

> Companion to **Chapter 9 — Hello, GPU** and **9a**.  Distilled from *CUDA by Example* (Sanders & Kandrot), chapter 4.

In 9a you launched a *single* thread (`<<<1, 1>>>`). Now we launch **many**. This chapter introduces the two ways CUDA spreads work across the hardware — **blocks** and **threads** — and then takes you into your first **2D grid** with a genuinely fun payoff: computing a **Julia-set fractal**, one pixel per block, entirely on the GPU, and displaying it here in the notebook.

### Learning objectives

By the end you will:

- Launch `N` **blocks** with `<<<N, 1>>>` and index data with `blockIdx.x` (the book's first parallel kernel).
- See how this relates to Ch9's `threadIdx.x` indexing — two axes of the same hierarchy.
- Use a **2D** launch geometry (`dim3`) and compute a 2D→1D offset `x + y * gridDim.x`.
- Compute and visualize the Julia set on the GPU — your first "real" parallel program with a picture to show for it.


## 1. From threads to blocks — vector addition

The "hello world" of data-parallel computing is adding two vectors: `c[i] = a[i] + b[i]`.

![Summing two vectors a and b element-wise into c](course/figures/fig_4_1_summing_vectors.png)

*Figure 4.1 (CUDA by Example): every element pairs up independently — there are no dependencies between positions, so all `N` additions can happen at once.*

In Ch9 you parallelized by launching many **threads** and indexing with `threadIdx.x`. *CUDA by Example* introduces parallelism the other way first — by launching many **blocks** of one thread each, indexed with `blockIdx.x`:

```c
add<<<N, 1>>>(dev_a, dev_b, dev_c);   //  N blocks, 1 thread per block
//        ^   ^
//        |   +-- blockDim = 1 thread
//        +------ gridDim  = N blocks   ->  blockIdx.x runs 0..N-1
```

Inside the kernel, `int tid = blockIdx.x;` gives each block its own element. Same result as Ch9's `threadIdx.x` — different axis of the same grid/block/thread hierarchy.


In [ ]:
!mkdir -p course/ch09b_build


In [ ]:
%%writefile course/ch09b_build/vector_add_blocks.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 10

// One BLOCK per element: tid comes from blockIdx.x, not threadIdx.x.
__global__ void add(int* a, int* b, int* c) {
    int tid = blockIdx.x;            // this block's element
    if (tid < N) c[tid] = a[tid] + b[tid];
}

int main(void) {
    int a[N], b[N], c[N];
    int *dev_a, *dev_b, *dev_c;
    cudaMalloc((void**)&dev_a, N*sizeof(int));
    cudaMalloc((void**)&dev_b, N*sizeof(int));
    cudaMalloc((void**)&dev_c, N*sizeof(int));

    for (int i = 0; i < N; i++) { a[i] = -i; b[i] = i*i; }   // fill on the host

    cudaMemcpy(dev_a, a, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(dev_b, b, N*sizeof(int), cudaMemcpyHostToDevice);

    add<<<N, 1>>>(dev_a, dev_b, dev_c);                       // N blocks, 1 thread each

    cudaMemcpy(c, dev_c, N*sizeof(int), cudaMemcpyDeviceToHost);

    int ok = 1;
    for (int i = 0; i < N; i++) {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
        if (c[i] != a[i] + b[i]) ok = 0;
    }
    printf("%s\n", ok ? "PASS" : "FAIL");

    cudaFree(dev_a); cudaFree(dev_b); cudaFree(dev_c);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch09b_build/vector_add_blocks course/ch09b_build/vector_add_blocks.cu && ./course/ch09b_build/vector_add_blocks


**Blocks vs threads — which should you use?** For now, both index a 1D array the same way:

| Axis | Built-in variable | Launch | Limit |
|---|---|---|---|
| Threads (Ch9) | `threadIdx.x` | `<<<1, N>>>` | ≤ 1024 per block |
| Blocks (here) | `blockIdx.x` | `<<<N, 1>>>` | up to 2³¹−1 blocks |

Neither alone scales: a single block caps at 1024 threads, and 1-thread blocks waste the hardware. **Real kernels combine them** — `blockIdx.x * blockDim.x + threadIdx.x`, exactly Ch9's formula. Chapter 10 (grid-stride loops) makes that combination handle *any* `N`. For now, the point is simply: **`blockIdx` and `threadIdx` are two coordinates into the same launch grid.**


## 2. A second dimension — `dim3` and 2D grids

So far every launch has been 1D (the `.x` axis only). But blocks and threads each have **`.x`, `.y`, and `.z`** coordinates. For anything naturally 2D — images, matrices — you launch a **2D grid** with `dim3`:

```c
dim3 grid(DIM, DIM);     //  DIM x DIM blocks
kernel<<<grid, 1>>>(...);
```

Now each block reads **two** coordinates, `blockIdx.x` and `blockIdx.y`:

![A 2D arrangement: rows of blocks, each block a row of threads](course/figures/fig_5_1_2d_blocks_threads.png)

*Figure 5.1 (CUDA by Example): a thread (or here, a block) is addressed by more than one index. The highlighted cell is `(blockIdx=2, thread=2)`.*

The one new idea is flattening a 2D coordinate into a 1D memory offset:

```c
int x = blockIdx.x;
int y = blockIdx.y;
int offset = x + y * gridDim.x;     //  row-major: skip y full rows, then add x
```

That's it. Let's use it to draw a fractal.


## 3. The Julia set — a 2D grid that produces a picture

The **Julia set** asks, for each point on the complex plane: does iterating `z → z² + c` stay bounded? Points that stay bounded are "in the set" (drawn bright); points that fly off to infinity are not (drawn dark). Every pixel is **independent** — a perfect data-parallel problem. We give each pixel its own block in a `DIM × DIM` grid.


In [ ]:
%%writefile course/ch09b_build/julia.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define DIM 1000

// A minimal complex number that works on the device.
struct cuComplex {
    float r, i;
    __device__ cuComplex(float a, float b) : r(a), i(b) {}
    __device__ float magnitude2(void) { return r*r + i*i; }
    __device__ cuComplex operator*(const cuComplex& a) {
        return cuComplex(r*a.r - i*a.i, i*a.r + r*a.i);
    }
    __device__ cuComplex operator+(const cuComplex& a) {
        return cuComplex(r + a.r, i + a.i);
    }
};

// Returns 1 if pixel (x,y) is in the Julia set, else 0.
__device__ int julia(int x, int y, float cr, float ci) {
    const float scale = 1.5f;
    float jx = scale * (float)(DIM/2 - x) / (DIM/2);
    float jy = scale * (float)(DIM/2 - y) / (DIM/2);
    cuComplex c(cr, ci);
    cuComplex a(jx, jy);
    for (int i = 0; i < 200; i++) {
        a = a * a + c;
        if (a.magnitude2() > 1000) return 0;   // escaped -> not in the set
    }
    return 1;
}

__global__ void kernel(unsigned char* ptr, float cr, float ci) {
    int x = blockIdx.x;                  // 2D grid: one block per pixel
    int y = blockIdx.y;
    int offset = x + y * gridDim.x;      // flatten 2D -> 1D
    ptr[offset] = (unsigned char)(255 * julia(x, y, cr, ci));
}

int main(int argc, char** argv) {
    float cr = (argc >= 3) ? atof(argv[1]) : -0.8f;
    float ci = (argc >= 3) ? atof(argv[2]) : 0.156f;
    const char* out = (argc >= 4) ? argv[3] : "course/ch09b_build/julia.bin";

    unsigned char* host_bitmap = (unsigned char*)malloc(DIM * DIM);
    unsigned char* dev_bitmap;
    cudaMalloc((void**)&dev_bitmap, DIM * DIM);

    dim3 grid(DIM, DIM);                 // DIM x DIM = 1,000,000 blocks
    kernel<<<grid, 1>>>(dev_bitmap, cr, ci);

    cudaMemcpy(host_bitmap, dev_bitmap, DIM * DIM, cudaMemcpyDeviceToHost);

    FILE* f = fopen(out, "wb");
    fwrite(host_bitmap, 1, DIM * DIM, f);
    fclose(f);
    printf("Computed %dx%d Julia set (c = %.3f + %.3fi) on a 2D grid of %d blocks -> %s\n",
           DIM, DIM, cr, ci, DIM*DIM, out);

    cudaFree(dev_bitmap); free(host_bitmap);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch09b_build/julia course/ch09b_build/julia.cu && ./course/ch09b_build/julia


In [ ]:
# Display the GPU-computed fractal. (Re-usable helper for the rest of the notebook.)
import numpy as np
import matplotlib.pyplot as plt

def show_bin(path, dim, cmap="magma", title=""):
    img = np.fromfile(path, dtype=np.uint8).reshape(dim, dim)
    plt.figure(figsize=(5.5, 5.5))
    plt.imshow(img, cmap=cmap); plt.axis("off")
    if title: plt.title(title)
    plt.show()
    return img

show_bin("course/ch09b_build/julia.bin", 1000, title="Julia set  (c = -0.8 + 0.156i) — computed on a 2D CUDA grid")

You just ran a kernel across a **1,000,000-block 2D grid**, and every block independently decided one pixel. The whole image came back in a single `cudaMemcpy`. That `offset = x + y * gridDim.x` line is the only new concept — everything else is the same five-step dance from 9a.


## 4. TODO Exercise — get the 2D offset right

Below, a kernel fills a `512 × 512` image with a smooth diagonal gradient — *if* you compute the flattened offset correctly. The `offset` line is left as `0`, so every block writes to pixel 0 and the image comes back **all black**. Fix the one line, recompile, and run the display cell to see the gradient appear.


In [ ]:
%%writefile course/ch09b_build/gradient.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define DIM 512

__global__ void grad(unsigned char* ptr) {
    int x = blockIdx.x;
    int y = blockIdx.y;

    // TODO: flatten the 2D coordinate (x, y) into a 1D offset (row-major).
    int offset = x + y * gridDim.x;   // <-- replace 0

    ptr[offset] = (unsigned char)((x + y) * 255 / (2 * DIM));
}

int main(void) {
    unsigned char* host = (unsigned char*)malloc(DIM * DIM);
    unsigned char* dev;
    cudaMalloc((void**)&dev, DIM * DIM);
    dim3 grid(DIM, DIM);
    grad<<<grid, 1>>>(dev);
    cudaMemcpy(host, dev, DIM * DIM, cudaMemcpyDeviceToHost);
    FILE* f = fopen("course/ch09b_build/gradient.bin", "wb");
    fwrite(host, 1, DIM * DIM, f); fclose(f);
    printf("Wrote course/ch09b_build/gradient.bin\n");
    cudaFree(dev); free(host);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch09b_build/gradient course/ch09b_build/gradient.cu && ./course/ch09b_build/gradient


In [ ]:
show_bin("course/ch09b_build/gradient.bin", 512, cmap="viridis", title="Your gradient (all black until the offset is fixed)")

### Solution

In [ ]:
%%writefile course/ch09b_build/gradient_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define DIM 512

__global__ void grad(unsigned char* ptr) {
    int x = blockIdx.x;
    int y = blockIdx.y;
    int offset = x + y * gridDim.x;        // row-major flatten
    ptr[offset] = (unsigned char)((x + y) * 255 / (2 * DIM));
}

int main(void) {
    unsigned char* host = (unsigned char*)malloc(DIM * DIM);
    unsigned char* dev;
    cudaMalloc((void**)&dev, DIM * DIM);
    dim3 grid(DIM, DIM);
    grad<<<grid, 1>>>(dev);
    cudaMemcpy(host, dev, DIM * DIM, cudaMemcpyDeviceToHost);
    FILE* f = fopen("course/ch09b_build/gradient_sol.bin", "wb");
    fwrite(host, 1, DIM * DIM, f); fclose(f);
    printf("Wrote course/ch09b_build/gradient_sol.bin\n");
    cudaFree(dev); free(host);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch09b_build/gradient_sol course/ch09b_build/gradient_sol.cu && ./course/ch09b_build/gradient_sol


In [ ]:
show_bin("course/ch09b_build/gradient_sol.bin", 512, cmap="viridis", title="Solution: offset = x + y * gridDim.x")

## 5. Explore — change the constant `c`, get a different fractal

The Julia program takes `c = cr + ci·i` on the command line. Different constants give wildly different shapes. Run a couple and re-display — this is the interactive payoff of having the whole thing on the GPU (each 1M-block render is near-instant).

In [ ]:
!./course/ch09b_build/julia 0.285 0.01 course/ch09b_build/julia2.bin


In [ ]:
show_bin("course/ch09b_build/julia2.bin", 1000, cmap="magma", title="Julia set  (c = 0.285 + 0.01i)")

## 6. The Translation Bridge

| Concept | Ch9 (threads) | Ch9b (blocks) | Ch9b (2D) |
|---|---|---|---|
| Launch | `<<<grid, block>>>` | `<<<N, 1>>>` | `<<<dim3(DIM,DIM), 1>>>` |
| Index | `threadIdx.x` (+ block) | `blockIdx.x` | `blockIdx.x`, `blockIdx.y` |
| 1D offset | `i` | `tid` | `x + y * gridDim.x` |
| Limit | 1024 threads/block | 2³¹−1 blocks | 65535 in y/z, 2³¹−1 in x |

The grid/block/thread hierarchy (Ch9's Figure 5.2) is just a coordinate system. `threadIdx` and `blockIdx`, each with `.x/.y/.z`, are the axes; a kernel's whole job is to turn its coordinates into the right memory offset.


## 7. Common Pitfalls

1. **Wrong 2D flatten.** `offset = x + y * gridDim.x` (row-major). Swapping to `y + x * gridDim.y` transposes the image; using the wrong dimension corrupts it. `gridDim.x` is the row length.
2. **Exceeding the y/z grid limit.** `blockIdx.y`/`.z` grids cap at **65535** (`blockIdx.x` goes to 2³¹−1). `DIM = 1000` is fine; a 100000-tall grid in `y` is not.
3. **One thread per block is wasteful.** The Julia kernel uses `<<<grid, 1>>>` for clarity, but it leaves 31/32 of every warp idle. Production code puts many threads per block (Ch10–11). It's correct, not fast.
4. **Forgetting the image is `unsigned char`.** Reading `julia.bin` as the wrong dtype/shape in NumPy gives garbage — it's `DIM*DIM` bytes, row-major.


## Recap

- A kernel launch has **two axes of parallelism**: a grid of **blocks** (`blockIdx`) and, within each, a set of **threads** (`threadIdx`). Each has `.x/.y/.z`. Ch9 used threads; here you used blocks; real kernels combine them.
- A **2D grid** (`dim3`) lets a kernel address 2D data directly. The only new line is the row-major flatten `offset = x + y * gridDim.x`.
- You computed a **Julia-set fractal** across a 1,000,000-block grid and displayed it — a real, visual, embarrassingly-parallel GPU program.

### What's next

**Chapter 10 — Grid-Stride Loops & Element-wise Ops.** We finally combine blocks *and* threads into `blockIdx.x * blockDim.x + threadIdx.x`, and add the **grid-stride loop** so one kernel handles any `N`, no matter how large — the pattern `llm.c`'s production kernels actually use.
